# Activity 1: RAGAS Evaluation with Cost Analysis

Compare two RAG pipelines:
- **Fireworks AI** — open-source model + Fireworks embeddings
- **OpenAI** — `gpt-4.1-mini` + `text-embedding-3-small`

We use 5 hand-written evaluation questions, run them through both pipelines, and score with RAGAS.
LangSmith tracing captures token usage and cost automatically.

## 1. Setup

In [1]:
import os
from dotenv import load_dotenv
from getpass import getpass

load_dotenv()

for key in ["FIREWORKS_API_KEY", "OPENAI_API_KEY", "LANGSMITH_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass(f"Enter {key}: ")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "activity-1-ragas-eval"

FIREWORKS_BASE_URL = "https://api.fireworks.ai/inference/v1"

## 2. Load & Split Documents

In [2]:
import tiktoken
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = PyMuPDFLoader("data/cat-health-guide.pdf").load()
print(f"Loaded {len(pages)} pages")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=750, chunk_overlap=0,
    length_function=lambda t: len(tiktoken.encoding_for_model("gpt-4o").encode(t)),
)
chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks")

Loaded 22 pages
Split into 42 chunks


## 3. Build RAG Pipelines

In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict

class State(TypedDict):
    question: str
    context: List[Document]
    response: str

RAG_PROMPT = ChatPromptTemplate.from_messages([("human",
    "\n#CONTEXT:\n{context}\n\nQUERY:\n{query}\n\n"
    "Use the provided context to answer the user query. "
    "Only use the provided context. If you don't know, say 'I don't know'."
)])

def build_rag(retriever, llm):
    def retrieve(state: State):
        return {"context": retriever.invoke(state["question"])}
    def generate(state: State):
        chain = RAG_PROMPT | llm | StrOutputParser()
        return {"response": chain.invoke({"query": state["question"], "context": state.get("context", [])})}
    g = StateGraph(State).add_sequence([retrieve, generate])
    g.add_edge(START, "retrieve")
    return g.compile()

In [4]:
# --- Pipeline A: Fireworks AI ---
fw_emb = OpenAIEmbeddings(
    model=os.environ.get("FIREWORKS_EMBEDDING_MODEL", "fireworks/qwen3-embedding-8b"),
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base=FIREWORKS_BASE_URL,
    check_embedding_ctx_length=False,
)
fw_vs = QdrantVectorStore.from_documents(chunks, fw_emb, location=":memory:", collection_name="fw", batch_size=10)
fw_llm = ChatOpenAI(
    model=os.environ.get("FIREWORKS_CHAT_MODEL", "accounts/fireworks/models/gpt-oss-20b"),
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base=FIREWORKS_BASE_URL,
)
fw_graph = build_rag(fw_vs.as_retriever(search_kwargs={"k": 3}), fw_llm)
print("✅ Fireworks pipeline ready")

# --- Pipeline B: OpenAI ---
oai_emb = OpenAIEmbeddings(model="text-embedding-3-small")
oai_vs = QdrantVectorStore.from_documents(chunks, oai_emb, location=":memory:", collection_name="oai")
oai_llm = ChatOpenAI(model="gpt-4.1-mini")
oai_graph = build_rag(oai_vs.as_retriever(search_kwargs={"k": 3}), oai_llm)
print("✅ OpenAI pipeline ready")

✅ Fireworks pipeline ready
✅ OpenAI pipeline ready


## 4. Evaluation Questions

We use 5 hand-crafted questions about the cat health guide, each with a reference answer.

In [5]:
EVAL_QA = [
    {
        "user_input": "What are the recommended veterinary visit frequencies for senior cats?",
        "reference": "Senior cats (aged 10+ years) should visit the veterinarian at least every 6 months for wellness exams.",
    },
    {
        "user_input": "What is the importance of feline-friendly handling during veterinary visits?",
        "reference": "Feline-friendly handling reduces stress for the cat and the owner, improves the quality of the exam, and increases the likelihood that owners will bring their cats in for regular veterinary visits.",
    },
    {
        "user_input": "What nutritional considerations are important for kittens?",
        "reference": "Kittens require diets that support rapid growth, including higher protein and calorie content. They should be fed a complete and balanced diet formulated for growth.",
    },
    {
        "user_input": "How does osteoarthritis affect senior cats?",
        "reference": "Osteoarthritis is common in senior cats and can cause chronic pain, reduced mobility, and behavioral changes. It often goes undetected because cats hide pain well.",
    },
    {
        "user_input": "What vaccinations are recommended for cats?",
        "reference": "Core vaccinations for cats include feline panleukopenia, feline herpesvirus, feline calicivirus, and rabies. Non-core vaccines depend on lifestyle and risk factors.",
    },
]

## 5. Run Both Pipelines

In [6]:
import time, copy

def run_pipeline(graph, qa_list, label, delay=2):
    results = []
    for i, qa in enumerate(qa_list):
        out = graph.invoke({"question": qa["user_input"]})
        results.append({
            "user_input": qa["user_input"],
            "reference": qa["reference"],
            "response": out["response"],
            "retrieved_contexts": [doc.page_content for doc in out["context"]],
        })
        if i < len(qa_list) - 1:
            time.sleep(delay)
    print(f"✅ {label}: {len(results)} queries done")
    return results

fw_results = run_pipeline(fw_graph, EVAL_QA, "Fireworks", delay=3)
oai_results = run_pipeline(oai_graph, EVAL_QA, "OpenAI", delay=1)

✅ Fireworks: 5 queries done
✅ OpenAI: 5 queries done


In [7]:
# Quick sanity check — first answer from each pipeline
print("Fireworks:", fw_results[0]["response"][:200])
print("\nOpenAI:", oai_results[0]["response"][:200])

Fireworks: Senior cats should be examined at least **every 6 months** (with veterinary visits more frequent if they have chronic conditions).

OpenAI: The recommended veterinary visit frequency for senior cats is at least every 6 months, with more frequent visits for those with chronic conditions.


## 6. Evaluate with RAGAS

In [8]:
from ragas import evaluate, EvaluationDataset, RunConfig
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy
from ragas.llms import llm_factory
from openai import OpenAI
import pandas as pd

evaluator_llm = llm_factory(model="gpt-4.1-mini", client=OpenAI(), max_tokens=16384)
evaluator_emb = OpenAIEmbeddings(model="text-embedding-3-small")
metrics = [LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy()]
run_cfg = RunConfig(timeout=360)

/var/folders/cc/j57vhbk52dnb9jpl23jvnxn80000gn/T/ipykernel_39053/2894120808.py:2: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy
/var/folders/cc/j57vhbk52dnb9jpl23jvnxn80000gn/T/ipykernel_39053/2894120808.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy
/var/folders/cc/j57vhbk52dnb9jpl23jvnxn80000gn/T/ipykernel_39053/2894120808.py:2: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please 

In [9]:
def to_eval_dataset(results):
    df = pd.DataFrame(results)
    return EvaluationDataset.from_pandas(df)

print("Evaluating Fireworks AI pipeline...")
fw_eval = evaluate(dataset=to_eval_dataset(fw_results), metrics=metrics,
                   llm=evaluator_llm, embeddings=evaluator_emb, run_config=run_cfg)
print(fw_eval)

print("\nEvaluating OpenAI pipeline...")
oai_eval = evaluate(dataset=to_eval_dataset(oai_results), metrics=metrics,
                    llm=evaluator_llm, embeddings=evaluator_emb, run_config=run_cfg)
print(oai_eval)

Evaluating Fireworks AI pipeline...


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'context_recall': 1.0000, 'faithfulness': 0.7580, 'factual_correctness(mode=f1)': 0.3900, 'answer_relevancy': 0.8687}

Evaluating OpenAI pipeline...


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'context_recall': 0.9000, 'faithfulness': 1.0000, 'factual_correctness(mode=f1)': 0.3480, 'answer_relevancy': 0.9210}


## 7. Side-by-Side Comparison

In [11]:
metric_keys = ["context_recall", "faithfulness", "factual_correctness(mode=f1)", "answer_relevancy"]

comparison = pd.DataFrame({
    "Metric": metric_keys,
    "Fireworks AI": [fw_eval[m] for m in metric_keys],
    "OpenAI": [oai_eval[m] for m in metric_keys],
})
comparison["Winner"] = comparison.apply(
    lambda r: "Tie" if r["Fireworks AI"] == r["OpenAI"]
    else ("Fireworks" if r["Fireworks AI"] > r["OpenAI"] else "OpenAI"), axis=1
)
comparison

,Metric,Fireworks AI,OpenAI,Winner
0,context_recall,"[1.0, 1.0, 1.0, 1.0, 1.0]","[1.0, 1.0, 1.0, 0.5, 1.0]",Fireworks
1,faithfulness,"[1.0, 0.7647058823529411, 0.5714285714285714, ...","[1.0, 1.0, 1.0, 1.0, 1.0]",OpenAI
2,factual_correctness(mode=f1),"[0.4, 0.45, 0.38, 0.36, 0.36]","[0.4, 0.42, 0.21, 0.36, 0.35]",Fireworks
3,answer_relevancy,"[0.8072866863704566, 0.9524562955963533, 0.888...","[0.8859496923345818, 0.9705058842580164, 0.982...",OpenAI


## 8. Cost Analysis via LangSmith

**LangSmith Tracing Screenshot (token usage & cost per run):**

![LangSmith Tracing - ChatOpenAI runs with token counts and cost](assets/image.png)

## Findings

**RAGAS Scores:**

| Metric | Fireworks AI | OpenAI |
|--------|---------------|--------|
| context_recall | 1.0000 | 0.9000 |
| faithfulness | 0.7580 | 1.0000 |
| factual_correctness (f1) | 0.3900 | 0.3480 |
| answer_relevancy | 0.8687 | 0.9210 |

**Cost & Performance (from LangSmith, AIE9-LangGraph-Local):**

| | Fireworks AI | OpenAI |
|---|--------------|--------|
| Cost/query | ~$0.0002–$0.0005 (est.) | $0.00086–$0.00172 |
| Tokens | 1,435–5,233 | 2,036–3,337 |
| Latency | 1.09s–10.74s | 1.09s–5.97s |
| Models | gpt-oss-20b, qwen3-embedding-8b | gpt-4.1-mini, text-embedding-3-small |

**Observations from LangSmith:** Fireworks runs show no cost in LangSmith (provider not supported). Some Fireworks responses returned "I don't know" (retrieval gaps); OpenAI responses were consistently substantive. Fireworks latency peaked at ~10.7s vs OpenAI ~6s.

**Fireworks Serverless vs Deploy:** Serverless ~$0.0003/query vs Deploy (A100) ~$0.04/query

**Summary:** Fireworks is ~3–5× cheaper (estimated); OpenAI leads on faithfulness, answer relevancy, and consistency. Choose Fireworks for cost; OpenAI for reliability and quality.